# Finite search caps and infinite-ridge certificates

The upper panel compares selected ridge penalties for caps $1,10^2,10^4,10^6$. The lower panel reports independent evidence at each sampled aspect ratio. Run all cells to load the included arrays, redraw the figure and save its PDF. Set `RECOMPUTE=True` to rerun the searches and certificates.

A **finite witness** has risk below $R_\infty$ by the stated tolerance. **Infinity certified** means a sufficient nonnegativity certificate covers every finite nonnegative penalty for that sampled $r$ and the decimalized parameters. **Unresolved** means neither conclusion was obtained. Following a search cap alone is not an infinity certificate; no conclusion is asserted between samples.


In [ ]:
from pathlib import Path
from dataclasses import asdict, replace
import importlib
import inspect
import json
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from copy import deepcopy

# Use only this distribution's local utilities, caches and output directories.
BUNDLE_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                    if (p/'dms'/'config.py').is_file()
                    and (p/'notebooks'/'06_infinite_ridge.ipynb').is_file()), None)
if BUNDLE_ROOT is None:
    raise RuntimeError('Start the kernel in the reproduction folder or its notebooks/ directory.')
foreign = [name for name, module in sys.modules.items()
           if (name == 'dms' or name.startswith('dms.'))
           and getattr(module, '__file__', None)
           and not Path(module.__file__).resolve().is_relative_to(BUNDLE_ROOT/'dms')]
if foreign:
    raise RuntimeError('Restart the kernel: another checkout of dms is already imported.')
sys.path.insert(0, str(BUNDLE_ROOT))
PROJECT_ROOT = BUNDLE_ROOT  # retained renderer name; always local to this bundle
OUTPUT_DIR = BUNDLE_ROOT/'output'
importlib.invalidate_caches()
for module_name in ('dms.config', 'dms.stieltjes', 'dms.simulation', 'dms.theory', 'dms.optimization', 'dms.ridge_diagnostics', 'dms.exploration', 'dms.result_cache', 'dms.infinity_diagnostics', 'dms.appendix_evidence'):
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
    else:
        importlib.import_module(module_name)
from dms.exploration import SweepParameters, parameter_sweep_cases
from dms.result_cache import ResultCache
from dms.appendix_evidence import cached_infinity_evidence
plt.rcParams.update({'axes.labelsize':11, 'axes.labelweight':'bold',
                     'xtick.labelsize':11, 'ytick.labelsize':11, 'legend.fontsize':11})
FIGURES = {}

from tools.portable_results import load_results, assert_settings_match

# False: reproduce from the included arrays. True: calculate from the inputs below.
RECOMPUTE = False


In [ ]:
# The parameter set of the corresponding appendix risk landscape.
BASE = SweepParameters(
    q=20, sigma=1., G2=100., f=.1, ftilde=.2, eta=.2,
    F=.4, FTest=.5, Fneq=.2, rhoneq=.1,
    m=.1, mneq=.2, mneqtilde=.3,
)
REFERENCE_L = 100
MEAN_CONVENTION = "normalized"
R_TRANSITION = np.linspace(1.7, 2.1, 81)
SEARCH_GRID = 641
CERTIFICATE_BRACKET_STEPS = 100

case, = parameter_sweep_cases(BASE, 'mneqtilde', [BASE.mneqtilde],
    mean_convention=MEAN_CONVENTION, reference_L=REFERENCE_L)
CFG = case.config
INFINITY_CAPS = [1., 1e2, 1e4, 1e6]
INFINITY_INPUTS = dict(config=CFG.as_dict(),
    # This figure requires cap searches and certificates only.
    r_slices=np.array([], dtype=float), lambda_grid=np.array([], dtype=float),
    r_transition=R_TRANSITION.copy(), caps=list(INFINITY_CAPS),
    n_grid=SEARCH_GRID, bracket_steps=CERTIFICATE_BRACKET_STEPS, witness_rtol=1e-10)
INFINITY_CACHE_OPTIONS = dict(directory=BUNDLE_ROOT/'.cache'/'infinite_ridge',
                              enabled=True, verbose=True)
INFINITY_FORCE_COMPUTE = False


## Compute searches and independent certificates

The producer uses 81 uniformly spaced aspect ratios on $[1.7,2.1]$, 641 samples
for each finite-cap search, and 100 exact-rational endpoint-bracketing steps.
The finite-witness threshold is $-10^{-10}R_\infty$. Coefficient certification is
sufficient, not necessary, and concerns the deterministic-equivalent risk for the
printed decimal parameters, including converted raw means.

The optimizer, sufficient certificate and witness evaluation use the stated
utility implementations. Plotting below reads this evidence without recalculation.

In [ ]:
if RECOMPUTE:
    INFINITY_EVIDENCE, INFINITY_EVIDENCE_ID = cached_infinity_evidence(
        ResultCache(**INFINITY_CACHE_OPTIONS), INFINITY_INPUTS,
        compute=True, force=INFINITY_FORCE_COMPUTE)
else:
    published, metadata = load_results(BUNDLE_ROOT/'data'/'infinite_ridge', restore_records=True)
    assert_settings_match(metadata['settings'], INFINITY_INPUTS)
    INFINITY_EVIDENCE, INFINITY_EVIDENCE_ID = published['evidence'], published['identity']
INFINITY_LOADED_INPUTS = deepcopy(INFINITY_INPUTS)
print({status: sum(row['status']==status for row in INFINITY_EVIDENCE['transition'])
       for status in ('finite witness','unresolved','infinity certified')})


## Editable renderer and presentation controls
The height ratio, independent ridge limits, colors, markers and shared aspect-ratio axis remain adjustable.

In [ ]:
def infinity_number(value):
    exponent = int(np.round(np.log10(value)))
    return fr'10^{{{exponent}}}' if value != 1 and np.isclose(value,10.**exponent) else f'{value:g}'


def draw_infinity_evidence(data, style):
    fig, axes = plt.subplots(2,1,sharex=True,
        figsize=(style['width'],style['height']),dpi=style['dpi'],layout='constrained',
        gridspec_kw=dict(height_ratios=style['height_ratios']))
    fig.get_layout_engine().set(**style['layout_padding'])
    upper, evidence = axes
    cap_colors = plt.get_cmap(style['cap_cmap'])(np.linspace(*style['cap_color_span'],len(data['caps'])))
    for (cap,opt),color in zip(data['caps'].items(),cap_colors):
        options = dict(color=color,linewidth=style['linewidth'],linestyle='-') | style['cap_styles'].get(cap,{})
        upper.plot(opt.r_values,opt.ridge_lambdas,
            label=style['cap_labels'].get(cap,fr'$\lambda_{{\max}}={infinity_number(cap)}$'),**options)
        if style['show_cap_markers']:
            hits = opt.status == 'upper'
            upper.plot(opt.r_values[hits],opt.ridge_lambdas[hits],
                       **(dict(color=options['color']) | style['cap_markers']))
    upper.set_yscale('symlog',linthresh=style['ridge_linthresh'])
    upper.set_ylim(bottom=0)
    statuses = ['finite witness','unresolved','infinity certified']
    for j,status in enumerate(statuses):
        rs = [row['r'] for row in data['transition'] if row['status']==status]
        evidence.plot(rs,np.full(len(rs),j),**style['status_styles'][status])
    evidence.set_yticks(range(len(statuses)),[style['status_labels'][s] for s in statuses])
    evidence.set_ylim(-.5,len(statuses)-.5)
    evidence.set_xscale(style['xscale'])
    if style['xlim'] is not None: evidence.set_xlim(style['xlim'])
    if style['xticks'] is not None: evidence.set_xticks(style['xticks'])
    evidence.set_xlabel(style['xlabel'])
    upper.tick_params(axis='x',which='both',labelbottom=False,bottom=style['show_top_x_ticks'])
    for i,ax in enumerate(axes):
        ax.set(ylabel=style['ylabels'][i],title=style['titles'][i])
        if style['ylims'][i] is not None: ax.set_ylim(style['ylims'][i])
        if style['yticks'][i] is not None: ax.set_yticks(style['yticks'][i])
        ax.tick_params(labelsize=style['tick_size'])
        for label in (ax.xaxis.label,ax.yaxis.label):
            label.set_fontsize(style['label_size']); label.set_fontweight('bold')
        ax.title.set_fontsize(style['title_size'])
        ax.grid(style['grid'])
        if style['panel_labels']:
            ax.text(*style['panel_label_xy'],chr(97+i),transform=ax.transAxes,
                    fontsize=style['panel_label_size'],fontweight='bold')
    if style['show_cap_legend']: upper.legend(**style['cap_legend'])
    for i,text in enumerate(style['annotations']):
        if text:
            axes[i].text(*style['annotation_xy'],text,transform=axes[i].transAxes,
                         ha='left',va='top',fontsize=style['label_size'])
    return fig


def plot_infinity_evidence(style, *, force=False):
    assert_settings_match(INFINITY_LOADED_INPUTS, INFINITY_INPUTS)
    data, identity = INFINITY_EVIDENCE, INFINITY_EVIDENCE_ID
    try:
        implementation = [inspect.getsource(f) for f in
                          (infinity_number,draw_infinity_evidence,plot_infinity_evidence)]
    except (OSError,TypeError):
        implementation,force = [],True
    fig, _ = ResultCache(**INFINITY_CACHE_OPTIONS).get('appendix_infinity_figure',
        dict(data=identity,style=style,rcparams={k:repr(v) for k,v in plt.rcParams.items()}),
        lambda:draw_infinity_evidence(data,style),implementation=implementation,force=force)
    FIGURES['appendix_infinite_ridge'] = fig
    display(fig);plt.close(fig)
    return fig


In [ ]:
# Presentation only: shared aspect-ratio axis, thin evidence panel; independent y quantities.
INFINITY_PLOT = dict(
    width=4.5,height=4.5,dpi=140,label_size=11,tick_size=11,title_size=11,
    panel_labels=True,panel_label_xy=(-.10,1.03),panel_label_size=12,
    layout_padding=dict(w_pad=.03,h_pad=.03,wspace=.06,hspace=.05),
    linewidth=1.8,grid=False,titles=['',''],height_ratios=(3.5,1.),
    cap_cmap='cividis',cap_color_span=(.15,.85),
    cap_styles={},cap_labels={},
    xlabel=r'Aspect ratio $r=qL/n$',xscale='linear',show_top_x_ticks=False,
    ylabels=[r'Best-found ridge $\lambda$', ''],
    xlim=None,ylims=[None,None],
    xticks=None,yticks=[None,None],
    ridge_linthresh=1e-6,
    show_cap_markers=True,
    cap_markers=dict(marker='^',markerfacecolor='none',markersize=4,linestyle='None'),
    status_labels={'finite witness':'Finite witness','unresolved':'Unresolved','infinity certified':'Infinity certified'},
    status_styles={
        'finite witness':dict(color='#3b528b',marker='o',markersize=3,linestyle='None'),
        'unresolved':dict(color='0.55',marker='x',markersize=4,linestyle='None'),
        'infinity certified':dict(color='#21918c',marker='|',markersize=7,linestyle='None')},
    show_cap_legend=True,cap_legend=dict(loc='best',fontsize=11,frameon=True),
    annotations=[None,None],annotation_xy=(.04,.96),
)
INFINITY_FORCE_REDRAW = False


## Draw the figure

This cell uses the loaded numerical arrays and the editable controls above.


In [ ]:
infinity_paper_fig = plot_infinity_evidence(INFINITY_PLOT,force=INFINITY_FORCE_REDRAW)


## Export the displayed figure
Saving does not evaluate the risk, rerun certificates or redraw.

In [ ]:
PDF_FIGURES = {'appendix_infinite_ridge': 'appendix_infinite_ridge.pdf'}
PDF_OUTPUT_DIR = OUTPUT_DIR
missing = [key for key in PDF_FIGURES if key not in FIGURES]
if missing: raise KeyError(f'Display these figures first: {missing}')
PDF_OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
for key,filename in PDF_FIGURES.items():
    output = PDF_OUTPUT_DIR/Path(filename).with_suffix('.pdf')
    FIGURES[key].savefig(output,format='pdf',bbox_inches='tight')
    print(output)
